# Kaggle E-commerce Gold Layer

This notebook creates analytics-ready gold tables from the cleaned silver layer.

**Source Schema:** `resume_project.default` (silver tables)  
**Target Schema:** `resume_project.gold`  

**Gold Tables Created:**
* `gold_hourly_sales` - Hourly order patterns
* `gold_daily_orders` - Daily (day of week) order patterns
* `gold_reorders` - Product reorder analysis
* `gold_departments` - Department performance metrics
* `gold_user_behavior` - Customer behavior and loyalty
* `gold_product_pairs` - Market basket analysis (products bought together)

In [0]:
CREATE OR REPLACE TABLE resume_project.gold.gold_hourly_sales AS
SELECT 
order_hour_of_day,
COUNT(order_id) AS Total_Sales_per_hour,
COUNT(DISTINCT user_id) AS Unique_Customers
FROM resume_project.default.orders
GROUP BY order_hour_of_day
ORDER BY order_hour_of_day



## Hourly Order Patterns
What time of day do customers order most?

## Daily Order Patterns (Day of Week)
Which days of the week see the most orders?

In [0]:
CREATE OR REPLACE TABLE resume_project.gold.gold_daily_orders AS
SELECT 
  order_dow,
  COUNT(order_id) AS Total_Sales_per_day,
  COUNT(DISTINCT user_id) AS Unique_Customers,
  ROUND(AVG(days_since_prior_order), 2) as avg_days_since_prior_order
FROM resume_project.default.orders
GROUP BY order_dow
ORDER BY order_dow

## Product Reorder Analysis
Which products are reordered the most?


In [0]:
CREATE OR REPLACE TABLE resume_project.gold.gold_reorders AS
SELECT P.product_name, SUM(PP.reordered) Times_Reordered, COUNT(PP.order_id) AS total_orders, ROUND((SUM(PP.reordered) / COUNT(PP.order_id)) * 100, 2) AS reorder_rate_percent
FROM resume_project.default.products P
JOIN resume_project.default.prodprior PP ON P.product_id = PP.product_id
GROUP BY P.product_name
ORDER BY COUNT(PP.order_id) DESC

## Department Analysis
Which departments are most popular? Which have the highest reorder rates?



In [0]:
CREATE OR REPLACE TABLE resume_project.gold.gold_departments AS
SELECT D.department_name, COUNT(PP.order_id) AS total_orders, COUNT(CASE WHEN PP.reordered = 1 THEN PP.order_id END) AS Reorders, ROUND((SUM(PP.reordered) / COUNT(PP.order_id)) * 100, 2) AS reorder_rate_percent
FROM resume_project.default.departments D
INNER JOIN resume_project.default.products P ON D.department_id = P.department_id
INNER JOIN resume_project.default.prodprior PP ON PP.product_id = P.product_id
GROUP BY D.department_name
ORDER BY COUNT(PP.order_id) DESC


## Customer Behavior Analysis
Which users reorder often? How frequently do customers place orders?


In [0]:
CREATE OR REPLACE TABLE resume_project.gold.gold_user_behavior AS
SELECT 
  O.user_id,
  COUNT(CASE WHEN P.reordered = 1 THEN P.order_id END) as total_reorders,
  COUNT(DISTINCT O.order_id) as total_orders,
  ROUND(AVG(O.days_since_prior_order), 2) as avg_days_between_orders,
  ROUND(COUNT(CASE WHEN P.reordered = 1 THEN P.order_id END) * 100.0 / COUNT(P.order_id), 2) as reorder_rate_percent
FROM resume_project.default.orders O
JOIN resume_project.default.prodprior P ON P.order_id = O.order_id
GROUP BY O.user_id
ORDER BY total_reorders DESC


## Product Affinity Analysis
Which products are commonly bought together (market basket analysis)?



In [0]:
CREATE OR REPLACE TABLE resume_project.gold.gold_product_pairs AS
SELECT 
  P1.product_name as product_1, 
  P2.product_name as product_2, 
  COUNT(*) as times_bought_together
FROM resume_project.default.prodprior PP1
JOIN resume_project.default.prodprior PP2 ON PP1.order_id = PP2.order_id
JOIN resume_project.default.products P1 ON PP1.product_id = P1.product_id
JOIN resume_project.default.products P2 ON PP2.product_id = P2.product_id
WHERE PP1.product_id < PP2.product_id
GROUP BY P1.product_name, P2.product_name
ORDER BY times_bought_together DESC
